# Start of Month Rebalance: 70% QQQ, 20% BIL, 10% GLD

This notebook runs a simple **start-of-month** rebalance strategy using TiPortfolio with data from **Alpaca** (optional cache via helpers). Set `APCA_API_KEY_ID` and `APCA_API_SECRET_KEY` in your environment or `.env` before running.

In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio.helpers.data import Alpaca, YFinance
from tiportfolio import (
    ScheduleBasedEngine,
    FixRatio,
    Schedule,
    compare_strategies,
    plot_strategy_comparison_interactive,
    rebalance_decisions_table,
)

from dotenv import load_dotenv
import os
load_dotenv()

alpaca = Alpaca(os.environ['ALPACA_API_KEY'], os.environ['ALPACA_API_SECRET'])

enable_data_source_cache("tiportfolio", cache_dir=".cache")

SYMBOLS = ["QQQ", "BIL", "GLD"]
WEIGHTS = {"QQQ": 0.7, "BIL": 0.2, "GLD": 0.1}
START = "2018-01-01"
END = "2024-12-31"
INITIAL_VALUE = 10_000

In [2]:
df = alpaca.query(SYMBOLS, START, END, timeframe="1d", adjust="all")
prices = {}
for symbol in df['symbol'].unique():
    sub = df[df['symbol'] == symbol].set_index('date')[['open', 'high', 'low', 'close']]
    prices[symbol] = sub
print(f"Loaded {len(prices)} symbols, {sum(len(v) for v in prices.values())} total rows")

Loaded cached bar data.

Loaded 3 symbols, 5280 total rows


In [3]:
engine = ScheduleBasedEngine(
    allocation=FixRatio(weights=WEIGHTS),
    rebalance=Schedule("month_start"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result = engine.run(symbols=SYMBOLS, start=START, end=END, prices_df=prices)
print(result.summary())

Backtest Summary
----------------
Sharpe Ratio:        0.6911
Sortino Ratio:       0.8997
MAR Ratio:           0.5824
CAGR:                15.33%
Max Drawdown:        26.31%
Kelly Leverage:      4.0709
Mean Excess Return:  0.1173
Final Value:         27,102.06
Total Fee:           0.76
Rebalances:          83


In [4]:

fig = result.plot_portfolio(mark_rebalances=True)
fig.show()

In [5]:
decisions_df = rebalance_decisions_table(result)
decisions_df.head(10)

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
0,2018-02-01 05:00:00+00:00,10444.058,10444.051,0.007,158.89,46.688,-0.676,46.012,7310.841,75.17,26.638,1.150,27.788,2088.812,128.07,7.990,0.165,8.155,1044.406
1,2018-03-01 05:00:00+00:00,10267.305,10267.302,0.003,155.60,46.012,0.178,46.190,7187.113,75.24,27.788,-0.496,27.292,2053.461,124.72,8.155,0.077,8.232,1026.730
2,2018-04-02 04:00:00+00:00,9910.068,9910.060,0.008,147.36,46.190,0.886,47.076,6937.048,75.33,27.292,-0.981,26.311,1982.014,127.26,8.232,-0.445,7.787,991.007
3,2018-05-01 04:00:00+00:00,10209.142,10209.135,0.007,154.25,47.076,-0.746,46.330,7146.399,75.42,26.311,0.762,27.073,2041.828,123.71,7.787,0.465,8.252,1020.914
4,2018-06-01 04:00:00+00:00,10639.572,10639.563,0.008,163.69,46.330,-0.831,45.499,7447.700,75.53,27.073,1.100,28.173,2127.914,122.51,8.252,0.432,8.685,1063.957
5,2018-07-02 04:00:00+00:00,10616.731,10616.729,0.002,164.09,45.499,-0.208,45.290,7431.712,75.63,28.173,-0.098,28.075,2123.346,117.46,8.685,0.354,9.039,1061.673
6,2018-08-01 04:00:00+00:00,10784.821,10784.817,0.004,168.19,45.290,-0.404,44.886,7549.375,75.75,28.075,0.399,28.475,2156.964,115.14,9.039,0.328,9.367,1078.482
7,2018-09-04 04:00:00+00:00,11139.643,11139.635,0.007,176.48,44.886,-0.701,44.185,7797.750,75.87,28.475,0.890,29.365,2227.929,112.93,9.367,0.497,9.864,1113.964
8,2018-10-01 04:00:00+00:00,11166.423,11166.422,0.000,177.10,44.185,-0.049,44.136,7816.496,75.97,29.365,0.032,29.397,2233.285,112.57,9.864,0.055,9.920,1116.642
9,2018-11-01 04:00:00+00:00,10618.506,10618.493,0.013,163.68,44.136,1.275,45.411,7432.954,76.11,29.397,-1.494,27.903,2123.701,116.63,9.920,-0.815,9.104,1061.851


In [6]:
decisions_df.tail(10)

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
73,2024-03-01 05:00:00+00:00,23573.746,23573.736,0.010,440.77,37.968,-0.530,37.438,16501.622,83.91,54.175,2.013,56.188,4714.749,192.89,11.887,0.334,12.221,2357.375
74,2024-04-01 04:00:00+00:00,23772.694,23772.689,0.005,440.70,37.438,0.322,37.760,16640.886,84.25,56.188,0.246,56.434,4754.539,207.82,12.221,-0.782,11.439,2377.269
75,2024-05-01 04:00:00+00:00,22984.889,22984.877,0.012,417.49,37.760,0.778,38.538,16089.422,84.61,56.434,-2.102,54.331,4596.978,213.79,11.439,-0.688,10.751,2298.489
76,2024-06-03 04:00:00+00:00,24249.594,24249.581,0.013,448.80,38.538,-0.716,37.822,16974.716,85.00,54.331,2.726,57.058,4849.919,217.22,10.751,0.412,11.164,2424.959
77,2024-07-01 04:00:00+00:00,25358.586,25358.573,0.013,478.08,37.822,-0.693,37.130,17751.010,85.35,57.058,2.365,59.423,5071.717,215.57,11.164,0.600,11.764,2535.859
78,2024-08-01 04:00:00+00:00,24681.328,24681.316,0.012,456.00,37.130,0.758,37.888,17276.930,85.73,59.423,-1.843,57.579,4936.266,225.77,11.764,-0.831,10.932,2468.133
79,2024-09-03 04:00:00+00:00,24835.050,24835.049,0.001,458.13,37.888,0.059,37.947,17384.535,86.14,57.579,0.083,57.662,4967.010,230.29,10.932,-0.148,10.784,2483.505
80,2024-10-01 04:00:00+00:00,25778.622,25778.614,0.009,478.11,37.947,-0.204,37.742,18045.036,86.49,57.662,1.949,59.611,5155.724,245.61,10.784,-0.288,10.496,2577.862
81,2024-11-01 04:00:00+00:00,26102.093,26102.091,0.002,484.22,37.742,-0.009,37.734,18271.465,86.84,59.611,0.505,60.115,5220.419,252.47,10.496,-0.157,10.339,2610.209
82,2024-12-02 05:00:00+00:00,27071.842,27071.830,0.012,511.90,37.734,-0.714,37.020,18950.290,87.15,60.115,2.012,62.127,5414.368,243.44,10.339,0.782,11.121,2707.184


## Comparison to Long QQQ

In [7]:
# Strategy 2: 100% QQQ (buy-and-hold, no rebalancing)
prices_qqq = {"QQQ": prices["QQQ"]}
engine_qqq = ScheduleBasedEngine(
    allocation=FixRatio(weights={"QQQ": 1.0}),
    rebalance=Schedule("month_start"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_qqq_only = engine_qqq.run(symbols=["QQQ"], start=START, end=END, prices_df=prices_qqq)
print("100% QQQ Summary")
print(result_qqq_only.summary())

100% QQQ Summary
Backtest Summary
----------------
Sharpe Ratio:        0.6860
Sortino Ratio:       0.8910
MAR Ratio:           0.5495
CAGR:                19.24%
Max Drawdown:        35.01%
Kelly Leverage:      2.8433
Mean Excess Return:  0.1655
Final Value:         34,218.64
Total Fee:           0.00
Rebalances:          0


In [8]:
# Compare: 70/20/10 rebalance vs 100% QQQ
comparison = compare_strategies(
    result,
    result_qqq_only,
    names=['rebalance', 'long qqq']
)
comparison

,rebalance,long qqq,better
metric,,,
sharpe_ratio,0.691077,0.685981,rebalance
sortino_ratio,0.899677,0.891008,rebalance
mar_ratio,0.582384,0.549477,rebalance
cagr,0.153252,0.192355,long qqq
max_drawdown,0.263145,0.350069,rebalance


In [9]:
# Strategy 3: Never rebalance 70/20/10 QQQ/BIL/GLD
engine_never = ScheduleBasedEngine(
    allocation=FixRatio(weights=WEIGHTS),
    rebalance=Schedule("never"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_never = engine_never.run(symbols=SYMBOLS, start=START, end=END, prices_df=prices)
print("Never Rebalance 70/20/10 Summary")
print(result_never.summary())

Never Rebalance 70/20/10 Summary
Backtest Summary
----------------
Sharpe Ratio:        0.6710
Sortino Ratio:       0.8757
MAR Ratio:           0.5304
CAGR:                15.99%
Max Drawdown:        30.14%
Kelly Leverage:      3.5626
Mean Excess Return:  0.1264
Final Value:         28,205.56
Total Fee:           0.00
Rebalances:          0


In [10]:
compare_strategies(result, result_qqq_only, result_never)

,Strategy 1,Strategy 2,Strategy 3,better
metric,,,,
sharpe_ratio,0.691077,0.685981,0.670993,Strategy 1
sortino_ratio,0.899677,0.891008,0.875660,Strategy 1
mar_ratio,0.582384,0.549477,0.530441,Strategy 1
cagr,0.153252,0.192355,0.159853,Strategy 2
max_drawdown,0.263145,0.350069,0.301358,Strategy 1


In [11]:
plot_strategy_comparison_interactive(
    result, result_qqq_only, result_never
)

## Comparing With Leverage

In [12]:
## Compare All Strategies

compare_strategies(result, result_qqq_only, result_never, leverages=[2, 1.4, 1.7], yearly_loan_rates=0.05)

,"Strategy 1 (L2x, r5.0%)","Strategy 2 (L1.4x, r5.0%)","Strategy 3 (L1.7x, r5.0%)",better
metric,,,,
sharpe_ratio,0.691077,0.685981,0.670993,"Strategy 1 (L2x, r5.0%)"
sortino_ratio,0.899677,0.891008,0.875660,"Strategy 1 (L2x, r5.0%)"
mar_ratio,0.487379,0.508668,0.462123,"Strategy 2 (L1.4x, r5.0%)"
cagr,0.256503,0.249296,0.236749,"Strategy 1 (L2x, r5.0%)"
max_drawdown,0.526291,0.490096,0.512309,"Strategy 2 (L1.4x, r5.0%)"


In [13]:
## Compare All Strategies

compare_strategies(result, result_qqq_only, result_never, leverages=[1.35, 1, 1.16], yearly_loan_rates=0.05)

,"Strategy 1 (L1.35x, r5.0%)",Strategy 2,"Strategy 3 (L1.16x, r5.0%)",better
metric,,,,
sharpe_ratio,0.691077,0.685981,0.670993,"Strategy 1 (L1.35x, r5.0%)"
sortino_ratio,0.899677,0.891008,0.875660,"Strategy 1 (L1.35x, r5.0%)"
mar_ratio,0.533122,0.549477,0.507556,Strategy 2
cagr,0.189390,0.192355,0.177429,Strategy 2
max_drawdown,0.355246,0.350069,0.349575,"Strategy 3 (L1.16x, r5.0%)"


In [14]:
plot_strategy_comparison_interactive(
    result, result_qqq_only, result_never,
    leverages=[2, 1.4, 1.7], yearly_loan_rates=0.05
)

## Portfolio Beta

Track portfolio beta over time vs SPY.


In [15]:
# Plot portfolio beta over time
fig_beta = result.plot_portfolio_beta()
fig_beta.show()


Loaded cached bar data.

